In [ ]:
# The supplied gene trees include 428 alignments with all 76 species.
library(RERconverge)
gene_tree_file = "related_files/new_gene_trees.tre"
gene_trees=readTrees(paste(gene_tree_file,sep = ""),minTreesAll = 428)


In [ ]:
rerpath = "related_files/"
fungi_name_all <- read.table(paste("../../1_data_acquisition_and_functional_annotation/related_files/76_fungi_trait.txt",sep = ""),header = TRUE)
fungi_name <- fungi_name_all$names
funRERw=getAllResiduals(gene_trees,useSpecies = fungi_name,transform = "sqrt",weighted = T,scale = T)


In [ ]:
outgroups <- c("Glomus_cerebriforme", "Umbelopsis_vinacea")
root_sp = outgroups
masterTree = gene_trees$masterTree


In [ ]:
all_yeast = c("Sporopachydermia_lactativora","Schizosaccharomyces_octosporus","Saccharomyces_cerevisiae","Lipomyces_kononenkoae","Saccharomycopsis_selenospora","Schizosaccharomyces_osmophilus","Candida_albicans","Yarrowia_lipolytica","Nakazawaea_anatomiae","Lachancea_nothofagi","Debaryomyces_udenii","Tortispora_agaves","Alloascoidea_africana","Schizosaccharomyces_cryophilus","Schizosaccharomyces_pombe","Lipomyces_chichibuensis", "Malassezia_sympodialis", "Pneumocystis_carinii", "Acaromyces_ingoldii", "Ustilago_maydis", "Kwoniella_shandongensis", "Kalmanozyma_brasiliensis", "Rhodotorula_diobovata", "Pneumocystis_jirovecii", "Kwoniella_pini", "Microbotryum_saponariae", "Taphrina_deformans", "Kwoniella_bestiolae", "Kwoniella_dejecticola", "Sporisorium_reilianum", "Malassezia_japonica", "Rhodotorula_toruloides", "Cutaneotrichosporon_oleaginosum", "Protomyces_lactucae", "Malassezia_restricta", "Cryptococcus_amylolentus", "Rhodotorula_graminis", "Exophiala_viscosa", "Moesziomyces_aphidis", "Rhodotorula_paludigena", "Aureobasidium_thailandense", "Aureobasidium_pullulans", "Aureobasidium_melanogenum")

yeastFg = all_yeast


all_sisters_yeast = list("clade1" = c("Schizosaccharomyces_octosporus", "Schizosaccharomyces_osmophilus", "Schizosaccharomyces_cryophilus", "Schizosaccharomyces_pombe", "Pneumocystis_jirovecii", "Pneumocystis_carinii", "Taphrina_deformans", "Protomyces_lactucae"),
                         "clade2" = c("Sporopachydermia_lactativora", "Saccharomyces_cerevisiae", "Lipomyces_kononenkoae", "Saccharomycopsis_selenospora", "Candida_albicans", "Yarrowia_lipolytica", "Nakazawaea_anatomiae", "Lachancea_nothofagi", "Debaryomyces_udenii", "Tortispora_agaves", "Alloascoidea_africana", "Lipomyces_chichibuensis"),
                         "clade3" = c("Microbotryum_saponariae", "Rhodotorula_toruloides", "Rhodotorula_paludigena", "Rhodotorula_graminis", "Rhodotorula_diobovata"),
                         "clade4" = c("Malassezia_japonica", "Malassezia_restricta", "Malassezia_sympodialis", "Sporisorium_reilianum", "Ustilago_maydis", "Kalmanozyma_brasiliensis", "Moesziomyces_aphidis"),
                         "clade5" = c("Cutaneotrichosporon_oleaginosum", "Cryptococcus_amylolentus", "Kwoniella_shandongensis", "Kwoniella_pini", "Kwoniella_bestiolae", "Kwoniella_dejecticola"),
                         "clade6" = c("Exophiala_viscosa"),
                         "clade7" = c("Aureobasidium_thailandense", "Aureobasidium_pullulans", "Aureobasidium_melanogenum"))


In [ ]:
options(repr.plot.width = 10, repr.plot.height = 15)
yeastFgTree = foreground2TreeClades(yeastFg, all_sisters_yeast, gene_trees, plotTree = F)
yeastFgTreerooted = root(yeastFgTree,outgroup = outgroups)
yeastplot1 = plotTreeHighlightBranches(yeastFgTreerooted,
                                       hlspecies = which(yeastFgTreerooted$edge.length==1),
                                       hlcols = "blue", main = "Species Tree\nForeground branches highlighted")


In [ ]:
phenvYeast = tree2PathsClades(yeastFgTree, gene_trees)


In [ ]:
corYeast=correlateWithBinaryPhenotype(funRERw,phenvYeast,min.sp = 10,min.pos = 2,weighted = "auto")


In [ ]:
corYeast$stat = -log10(corYeast$P) * sign(corYeast$Rho)
head(corYeast[order(corYeast$stat), ])
summary(corYeast$stat)


In [ ]:
# Candidate sets use raw P values; BH-adjusted values remain in the output.
corYeast_clean <- corYeast[!is.na(corYeast$Rho) & !is.na(corYeast$p.adj), ]
export_correlations <- function(result, filename) {
    result <- data.frame(gene = rownames(result), result, row.names = NULL)
    write.table(result, file = file.path(rerpath, filename),
                sep = "\t", row.names = FALSE, quote = FALSE)
}
filtered_positive <- corYeast_clean[corYeast_clean$Rho > 0.35 & corYeast_clean$P < 0.05, ]
filtered_negative <- corYeast_clean[corYeast_clean$Rho < -0.35 & corYeast_clean$P < 0.05, ]
export_correlations(filtered_positive, "cor_pos.txt")
export_correlations(filtered_negative, "cor_neg.txt")


In [ ]:
export_correlations(corYeast_clean, "cor_all.txt")


In [ ]:
rer_output_file = paste(rerpath, "funRERw_matrix.txt", sep = "")

write.table(funRERw,
            file = rer_output_file,
            sep = "\t",
            row.names = TRUE,
            col.names = NA,
            quote = FALSE)
